In [0]:

%run ../Includes/Copy-Datasets


In [0]:
%sql
alter table orders_silver add constraint timestamp_within_range check (order_timestamp >= '2020-01-01')

In [0]:
%sql
describe extended orders_silver; 

In [0]:
%sql
INSERT INTO orders_silver
VALUES ('1', '2022-02-01 00:00:00.000', 'C00001', 0, 0, NULL),
       ('2', '2019-05-01 00:00:00.000', 'C00001', 0, 0, NULL),
       ('3', '2023-01-01 00:00:00.000', 'C00001', 0, 0, NULL);

In [0]:
%sql
select * from orders_silver where order_id in ('1','2','3');

In [0]:
%sql
alter table orders_silver add constraint valid_quantity check (quantity > 0);

In [0]:
%sql 
select *
from orders_silver
where quantity <= 0;

In [0]:
from pyspark.sql import functions as F

json_schema = "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"

query = (spark.readStream.table("bronze")
                .where(F.col("topic") == "orders")
                .select(F.from_json(F.col("value").cast("string"), json_schema).alias("data"))
                .select("data.*")
                .filter(F.col("quantity") > 0)
            .writeStream
                .option("checkpointLocation", f"{bookstore.checkpoint_path}/orders_silver")
                .trigger(availableNow=True)
                .table("orders_silver")
         )
query.awaitTermination()

In [0]:
%sql
select count(*)
from orders_silver
where quantity <= 0;
    

In [0]:
%sql
alter table orders_silver drop constraint timestamp_within_range;

In [0]:
%sql
describe extended orders_silver;

In [0]:
%sql
drop table orders_silver;


In [0]:
dbutils.fs.rm(f"{bookstore.checkpoint_path}/Orders_silver", True)
dbutils.fs.rm(f"{bookstore.checkpoint_path}/orders_silver", True)